In [1]:
%pip install -q langchain langchain-core langchain-text-splitters langchain-groq\
langchain-huggingface langchain-chroma arxiv chromadb pymupdf sentence-transformers\
langsmith requests pydantic pandas langchain-community


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [2]:

%pip install -q \
    requests==2.32.4 \
    python-dateutil==2.9.0.post0 \
    opentelemetry-api==1.42.1 \
    opentelemetry-sdk==1.42.1 \
    oauthlib==3.2.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-grpc 1.45.2 requires opentelemetry-sdk~=1.45.2, but you have opentelemetry-sdk 1.42.1 which is incompatible.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.
opentelemetry-exporter-otlp-common 0.66b2 requires opentelemetry-sdk~=1.45.2, but you have opentelemetry-sdk 1.42.1 which is incompatible.
kubernetes 37.0.3 requires oauthlib>=4.0.0, but you have oauthlib 3.2.2 which is incompatible.


In [3]:
%pip install requests

In [4]:

import requests
import dateutil
import opentelemetry
import chromadb
import langchain
import torch

print("Requests:", requests.__version__)
print("Dateutil:", dateutil.__version__)
print("ChromaDB:", chromadb.__version__)
print("LangChain:", langchain.__version__)
print("CUDA available:", torch.cuda.is_available())

print("\nEnvironment verification completed.")


Requests: 2.32.4
Dateutil: 2.9.0.post0
ChromaDB: 1.5.9
LangChain: 1.4.8
CUDA available: False

Environment verification completed.


In [5]:

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter

import arxiv
import fitz
import sentence_transformers
import langsmith

print("Groq integration: OK")
print("Hugging Face integration: OK")
print("Chroma integration: OK")
print("Text splitter: OK")
print("arXiv: OK")
print("PyMuPDF: OK")
print("Sentence Transformers: OK")
print("LangSmith: OK")

print("\nAll ScholarPilot integrations verified!")


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Groq integration: OK
Hugging Face integration: OK
Chroma integration: OK
Text splitter: OK
arXiv: OK
PyMuPDF: OK
Sentence Transformers: OK
LangSmith: OK

All ScholarPilot integrations verified!


In [ ]:
import os, getpass           

def get_secret(name):         
    
    try:
        from google.colab import userdata      
        return userdata.get(name)              
    except Exception:                          
        
        return os.environ.get(name) or getpass.getpass(f"Enter {name}: ")


os.environ["GROQ_API_KEY"] = get_secret("GROQ_API_KEY")            
os.environ["LANGSMITH_API_KEY"] = get_secret("LANGSMITH_API_KEY")  
os.environ["LANGSMITH_TRACING"] = "true"          
os.environ["LANGSMITH_PROJECT"] = "scholarpilot"  

print("Keys loaded ✅")       

Keys loaded ✅


In [7]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0
)

reply = llm.invoke(
    "In one sentence, What is RAG in AI"
)
print(reply.content)

RAG (Retrieval‑Augmented Generation) is an AI approach that combines a language model with a real‑time search of external documents or databases, retrieving relevant information to feed into the model so it can generate more accurate, up‑to‑date responses.


In [8]:
from dataclasses import dataclass

@dataclass
class Paper:
  title : str
  abstract : str
  year : str
  url : str
  source : str
  pdf_url : str = ""

In [9]:

import arxiv
from langsmith import traceable

@traceable(name="search_arxiv")
def search_arxiv(query: str, k: int = 5) -> list[Paper]:

    try:
        # Create the arXiv API client
        client = arxiv.Client()

        # Prepare the search request
        search = arxiv.Search(
            query=query,
            max_results=k,
            sort_by=arxiv.SortCriterion.Relevance
        )

        # Execute the search using the Client
        results = client.results(search)

        papers = []

        # Convert each arXiv result into our Paper dataclass
        for result in results:

            papers.append(Paper(
                title=result.title,
                abstract=result.summary.replace("\n", " "),
                year=str(result.published.year),
                url=result.entry_id,
                source="arXiv",
                pdf_url=result.pdf_url,
            ))

        return papers

    except Exception as e:
        print(f"[arXiv] failed: {e}")
        return []


# Test the arXiv search function
results = search_arxiv(
    "retrieval augmented generation hallucination",
    k=3
)

if results:
    for p in results:
        print(p.year, "|", p.title)
        print("   PDF:", p.pdf_url)
        print()
else:
    print("No papers returned. Check the search or connection.")


2025 | AR-RAG: Autoregressive Retrieval Augmentation for Image Generation
   PDF: https://arxiv.org/pdf/2506.06962v3

2025 | Intelligent Interaction Strategies for Context-Aware Cognitive Augmentation
   PDF: https://arxiv.org/pdf/2504.13684v1

2025 | Factually: Exploring Wearable Fact-Checking for Augmented Truth Discernment
   PDF: https://arxiv.org/pdf/2504.17204v1



In [10]:

import requests
from google.colab import userdata

# Load the API key securely from Colab Secrets
OPENALEX_API_KEY = userdata.get("OPENALEX_API_KEY")

# OpenAlex works endpoint
url = "https://api.openalex.org/works"

# Search parameters
params = {
    "search": "retrieval augmented generation hallucination",
    "per_page": 3,
    "select": "id,title,publication_year,doi"
}

# Authenticate using the API key
headers = {
    "Authorization": f"Bearer {OPENALEX_API_KEY}"
}

# Send the request
response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=30
)

# Check the response
print("HTTP Status:", response.status_code)

response.raise_for_status()

data = response.json()

print("Total matching works:", data["meta"]["count"])
print("\nRetrieved papers:\n")

for paper in data["results"]:
    print("Title:", paper.get("title"))
    print("Year:", paper.get("publication_year"))
    print("DOI:", paper.get("doi"))
    print("-" * 60)


HTTP Status: 200
Total matching works: 42892

Retrieved papers:

Title: Reducing hallucination in structured outputs via Retrieval-Augmented Generation
Year: 2024
DOI: https://doi.org/10.18653/v1/2024.naacl-industry.19
------------------------------------------------------------
Title: ReDeEP: Detecting Hallucination in Retrieval-Augmented Generation via Mechanistic Interpretability
Year: 2024
DOI: https://doi.org/10.48550/arxiv.2410.11414
------------------------------------------------------------
Title: RAG-HAT: A Hallucination-Aware Tuning Pipeline for LLM in Retrieval-Augmented Generation
Year: 2024
DOI: https://doi.org/10.18653/v1/2024.emnlp-industry.113
------------------------------------------------------------


In [11]:

import requests
import time
from langsmith import traceable


def reconstruct_abstract(inverted_index):
    """
    Convert OpenAlex's abstract inverted index
    into normal readable text.
    """

    if not inverted_index:
        return ""

    # Find the total number of word positions
    max_position = max(
        position
        for positions in inverted_index.values()
        for position in positions
    )

    # Create an empty list for the words
    words = [""] * (max_position + 1)

    # Put each word in its original position
    for word, positions in inverted_index.items():
        for position in positions:
            words[position] = word

    # Join words into a normal sentence
    return " ".join(word for word in words if word)


@traceable(name="search_openalex")
def search_openalex(query: str, k: int = 5) -> list[Paper]:

    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "per_page": k,
        "select": (
            "id,title,publication_year,doi,"
            "abstract_inverted_index,best_oa_location,primary_location"
        )
    }

    headers = {
        "Authorization": f"Bearer {OPENALEX_API_KEY}"
    }

    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=30
        )

        response.raise_for_status()

        data = response.json()

    except requests.exceptions.RequestException as e:
        print(f"[OpenAlex] Request failed: {e}")
        return []

    papers = []

    for item in data.get("results", []):

        # Reconstruct the abstract
        abstract = reconstruct_abstract(
            item.get("abstract_inverted_index")
        )

        # Skip papers without abstracts
        if not abstract:
            continue

        # Find an available PDF URL
        best_location = item.get("best_oa_location") or {}
        primary_location = item.get("primary_location") or {}

        pdf_url = (
            best_location.get("pdf_url")
            or primary_location.get("pdf_url")
            or ""
        )

        # Use DOI as the paper URL when available
        paper_url = item.get("doi") or item.get("id") or ""

        papers.append(Paper(
            title=item.get("title") or "Unknown title",
            abstract=abstract,
            year=str(item.get("publication_year") or ""),
            url=paper_url,
            source="OpenAlex",
            pdf_url=pdf_url
        ))

    return papers


# Test the function
results = search_openalex(
    "retrieval augmented generation hallucination",
    k=3
)

if results:
    for p in results:
        print(p.year, "|", p.title)
        print("Source:", p.source)
        print("Abstract:", p.abstract[:250], "...")
        print("PDF available:", bool(p.pdf_url))
        print()
else:
    print("No papers returned. Check the search or connection.")


2024 | Reducing hallucination in structured outputs via Retrieval-Augmented Generation
Source: OpenAlex
Abstract: Orlando Ayala, Patrice Bechard. Proceedings of the 2024 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Technologies (Volume 6: Industry Track). 2024. ...
PDF available: True

2024 | ReDeEP: Detecting Hallucination in Retrieval-Augmented Generation via Mechanistic Interpretability
Source: OpenAlex
Abstract: Retrieval-Augmented Generation (RAG) models are designed to incorporate external knowledge, reducing hallucinations caused by insufficient parametric (internal) knowledge. However, even with accurate and relevant retrieved content, RAG models can sti ...
PDF available: True

2024 | RAG-HAT: A Hallucination-Aware Tuning Pipeline for LLM in Retrieval-Augmented Generation
Source: OpenAlex
Abstract: Retrieval-augmented generation (RAG) has emerged as a significant advancement in the field of large language models (LLM

In [12]:

from langsmith import traceable

@traceable(name="search_all_sources")
def search_all(queries: list[str], k: int = 4) -> list[Paper]:

    found = []
    seen = set()

    for q in queries:

        # Search both currently working sources
        arxiv_papers = search_arxiv(q, k)
        openalex_papers = search_openalex(q, k)

        # Combine the results
        combined_papers = arxiv_papers + openalex_papers

        # Remove duplicate titles
        for p in combined_papers:

            key = p.title.lower().strip()

            if key not in seen:
                seen.add(key)
                found.append(p)

    return found


In [13]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

sentences = [
    "How to reduce hallucination in language models",
    "Methods to mitigate made-up answers from LLMs",
    "Best recipe for chocolate cake",

]
vecs = [np.array(embeddings.embed_query(s)) for s in sentences]

def cosine(a, b):
  return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("vector length:", len(vecs[0]))
print("first 5 numbers:", vecs[0][:5].round(3))
print("A vs B (same meaning):", round(cosine(vecs[0], vecs[1]), 2))
print("A vs C (unrelated)   :", round(cosine(vecs[0], vecs[2]), 2))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

vector length: 384
first 5 numbers: [-0.025  0.05   0.046  0.012 -0.026]
A vs B (same meaning): 0.64
A vs C (unrelated)   : 0.46


In [14]:
import hashlib
from langchain_chroma import Chroma
from langchain_core.documents import Document

vectorstore = Chroma(collection_name="papers", embedding_function=embeddings)

def title_hash(title: str) -> str:
  return hashlib.md5(title.lower().encode()).hexdigest()

def index_papers(papers: list[Paper]):
  docs = [
      Document(
          page_content=f"{p.title}. {p.abstract}",
          metadata={"title": p.title, "year": p.year,
                    "url": p.url, "source": p.source,
                    "pdf_url": p.pdf_url},
      )
      for p in papers
  ]
  if docs:
    vectorstore.add_documents(docs, ids=[title_hash(p.title) for p in papers])

@traceable(name="retrive_from_index")
def retrieve(question: str, k: int = 8) -> list[Document]:
  return vectorstore.similarity_search(question, k=k)

In [15]:
from pydantic import AliasChoices
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


# 1. Structure for the research planner
class SearchPlan(BaseModel):
    queries: list[str] = Field(
        description="2 to 3 short keyword queries for academic paper search"
    )


# 2. Structure for the paper relevance grader
class Grade(BaseModel):
    relevant: bool = Field(
        description="True if the paper helps answer the question"
    )


# 3. Structure for deciding whether abstracts are sufficient
class Sufficiency(BaseModel):
    enough: bool = Field(
        description="True if the abstracts alone contain enough detail to answer well"
    )


# 4. Structure for verifying the final answer


class Verdict(BaseModel):
    grounded: bool = Field(
        description="True if every claim in the answer is supported by the sources"
    )
    problems: str = Field(
        validation_alias=AliasChoices("problems", "problem"),
        description="Short description of unsupported claims, or empty text if none"
    )

# 5. Prompt for the research planner
planner_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You help search academic papers. Turn the research question "
        "into 2 to 3 short keyword queries for arXiv and OpenAlex. "
        "Return valid JSON only, with exactly one key: "
        '"queries", whose value is an array of strings. '
        "Do not include explanations or Markdown."
    ),
    ("human", "Research question: {question}"),
])

planner = planner_prompt | llm.with_structured_output(
    SearchPlan,
    method="json_mode"
)



# 7. Test the planner
result = planner.invoke({
    "question": "How can we reduce hallucinations in RAG systems?"
})

print("Generated search queries:")
for query in result.queries:
    print("-", query)


Generated search queries:
- retrieval augmented generation hallucination mitigation
- RAG hallucination reduction
- hallucination control language models


In [16]:


grader_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You judge whether a research paper is relevant to a question. Be reasonably strict."
    ),
    (
        "human",
        "Question: {question}\n\nPaper:\n{paper}\n\nIs this paper relevant?"
    ),
])

grader = grader_prompt | llm.with_structured_output(Grade)

@traceable(name="grade_documents")
def grade_documents(question: str, docs: list[Document]) -> list[Document]:
  inputs = [{"question": question, "paper": d.page_content[:1500]} for d in docs]
  grades = grader.batch(inputs, config={"max_concurrency": 3})
  return [d for d, g in zip(docs, grades) if g.relevant]

In [17]:
rewriter_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an academic search query optimizer. "
        "Generate 2 to 3 NEW keyword search queries. "
        "Use different wording, synonyms, and related concepts. "
        "Return valid JSON only, with exactly one key: "
        '"queries", whose value is an array of strings. '
        "Do not include explanations or Markdown."
    ),
    (
        "human",
        "Research question: {question}\n"
        "Queries already tried: {tried}"
    ),
])

rewriter = rewriter_prompt | llm.with_structured_output(
    SearchPlan,
    method="json_mode"
)

In [29]:
sufficiency_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Decide whether the paper ABSTRACTS alone contain enough detail "
        "to answer the question well. Broad overview questions can be "
        "answered from abstracts. Questions needing specific methods, "
        "numbers, comparisons or results usually need the full papers. "
        "Return valid JSON only, with exactly one key: "
        '"enough" (boolean). Do not include explanations or Markdown.'
    ),
    (
        "human",
        "Question: {question}\n\nAbstracts:\n{abstracts}"
    ),
])

sufficiency_checker = sufficiency_prompt | llm.with_structured_output(
    Sufficiency,
    method="json_mode"
)

In [19]:
HEADERS = {"User-Agent": "ScholarPilot/1.0 (learning project)"}

def download_pdf(pdf_url: str):
  try:
    r = requests.get(pdf_url, timeout=60, headers=HEADERS)
  except Exception as e:
    print(f"[PDF] download failed: {e}")
    return None

  if r.status_code != 200 or not r.content.startswith(b"%PDF"):
    print(f"[PDF] not a valid PDF (HTTP {r.status_code}): {pdf_url}")
    return None
  return r.content


In [20]:
import pymupdf

def extract_pages(pdf_bytes: bytes, max_pages: int = 12):
  doc = pymupdf.open(stram=pdf_bytes, filetype="pdf")
  pages = []
  for page_no, page in enumerate(doc, start=1):
    if page_no > max_pages:
      break
    text = " ".join(page.get_text().split())
    if text:
      pages.append((page_no, text))
  return pages

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def chunk_pages(pages, meta: dict):
  chunks = []
  for page_no, text in pages:
    chumks += splitter.create_documents([text], metadatas=[{**meta, "page": page_no}])
  return chunks

In [22]:
fulltext_store = Chroma(collection_name="fulltext", embedding_function=embeddings)
fulltext_done = set()

@traceable(name="load_fulltext")
def load_fulltext(paper_doc: Document) -> bool:
  m = paper_doc.metadata
  if m["title"] in fulltext_done:
    return True
  if not m.get("pdf_url"):
    print(f"[PDF] no pdf link: {m['title'][:60]}")
    return False
  pdf = download_pdf(m["pdf_url"])
  if pdf is None:
    return False
  pages = extract_pages(pdf)
  if not pages:
    print(f"[PDF] no text extracted: {m['title'][:60]}")
    return False
  meta = {"title": m["title"], "year": m["year"], "url": m["url"], "source": m["source"]}
  chunks = chunk_pages(pages, meta)
  ids = [f"{title_hash(m['title'])}-{i}" for i in range(len(chunks))]
  fulltext_store.add_documents(chunks, ids=ids)
  fulltext_done.add(m["title"])
  print(f"[PDF] loaded {len(pages)} pages, {len(chunks)} chunks: {m['title'][:60]}")
  return True

@traceable(name="retrieve_chunks")
def retrieve_chunks(question: str, titles: list[str], k: int = 6) -> list[Document]:
  return fulltext_store.similarity_search(question, k=k, filter={"title": {"$in": titles}})

In [23]:
def format_sources(docs: list[Document]) -> str:
    lines = []
    for i, d in enumerate(docs, start=1):
        m = d.metadata
        page = f", p.{m['page']}" if "page" in m else ""
        lines.append(f"[{i}] {m['title']} ({m['year']}, {m['source']}{page})\n{d.page_content[:1200]}")
    return "\n\n".join(lines)

def format_references(docs: list[Document]) -> str:
    lines = []
    for i, d in enumerate(docs, start=1):
        m = d.metadata
        page = f", p.{m['page']}" if "page" in m else ""
        lines.append(f"[{i}] {m['title']} ({m['year']}{page}) - {m['url']}")
    return "\n".join(lines)

def unique_titles(docs: list[Document]) -> list[str]:
    return list(dict.fromkeys(d.metadata["title"] for d in docs))

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a careful research assistant. Write a short literature-review style answer "
               "(about 150-250 words) using ONLY the numbered sources. Cite with [1], [2] after the "
               "claims they support. If the sources are not enough, say so. Do not invent facts. "
               "{feedback}"),
    ("human", "Question: {question}\n\nSources:\n{sources}"),
])
answer_chain = answer_prompt | llm

verify_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a strict fact-checker. Check whether the answer's claims "
        "are supported by the sources. Return valid JSON only, with exactly "
        "two fields: grounded (boolean) and problems (string). "
        "If all claims are supported, set grounded to true and problems to an empty string."
    ),
    (
        "human",
        "Sources:\n{sources}\n\nAnswer:\n{answer}"
    ),
])

verifier = verify_prompt | llm.with_structured_output(
    Verdict,
    method="json_mode"
)


In [26]:
MAX_ROUNDS = 2
MIN_RELEVANT = 3

@traceable(name="scholarpilot_agent")
def research_agent(question: str, use_pdfs: bool = True, max_pdfs: int = 2) -> dict:
  queries = planner.invoke({"question": question}).queries
  all_tried = list(queries)
  relevant = []

  for round_no in range(1, MAX_ROUNDS + 1):
    print(f"Round {round_no}: searching {queries}")
    index_papers(search_all(queries))
    candidates = retrieve(question)
    relevant = grade_documents(question, candidates)
    print(f" {len(relevant)}/{len(candidates)} papers judged relevant")

    if len(relevant) >= MIN_RELEVANT:
      break
    if round_no < MAX_ROUNDS:
      queries = rewriter.invoke({"question": question, "tried": all_tried}).queries
      all_tried += queries

  if not relevant:
    return {"answer": "I could not find relevant papers.", "sources": [],
            "grounded": False, "rounds": round_no, "used_pdf": False}

  # evidence = relevant
  # used_pdf = False
  # if use_pdfs:
  #     abstracts = format_sources(relevant)
  #     enough = sufficiency_checker.invoke(
  #         {"question": question, "abstracts": abstracts}).enough
  #     print(f"  abstracts enough? {enough}")
  #     if not enough:
  #         loaded = []
  #         for d in relevant:
  #             if len(loaded) >= max_pdfs:
  #                 break
  #             if load_fulltext(d):
  #                 loaded.append(d.metadata["title"])
  #         if loaded:
  #             chunks = retrieve_chunks(question, loaded)
  #             if chunks:
  #                 evidence, used_pdf = chunks, True

  evidence = relevant
  used_pdf = False

  if use_pdfs:
    #print("[DEBUG 1] Checking abstract sufficiency...")

    abstracts = format_sources(relevant)

    enough = sufficiency_checker.invoke(
        {"question": question, "abstracts": abstracts}
    ).enough

    #print(f"[DEBUG 2] Abstracts enough? {enough}")

    if not enough:
      loaded = []

      for d in relevant:
        if len(loaded) >= max_pdfs:
          break

        if load_fulltext(d):
          loaded.append(d.metadata["title"])

      if loaded:
        chunks = retrieve_chunks(question, loaded)

        if chunks:
          evidence, used_pdf = chunks, True

  #print("[DEBUG 3] Preparing answer...")
  sources_text = format_sources(evidence)

  #print("[DEBUG 4] Generating answer...")
  answer = answer_chain.invoke(
      {"question": question, "sources": sources_text, "feedback": ""}
  ).content

  #print("[DEBUG 5] Verifying answer...")
  verdict = verifier.invoke(
      {"sources": sources_text, "answer": answer}
  )

  #print("[DEBUG 6] Verification completed.")

  if not verdict.grounded:
    print("  Verifier found problems -> rewriting once:", verdict.problems)

    answer = answer_chain.invoke({
        "question": question,
        "sources": sources_text,
        "feedback": f"A fact-checker found these problems, fix them: {verdict.problems}",
    }).content

    verdict = verifier.invoke(
        {"sources": sources_text, "answer": answer}
    )

  return {
      "answer": answer + "\n\nReferences:\n" + format_references(evidence),
      "sources": unique_titles(evidence),
      "grounded": verdict.grounded,
      "rounds": round_no,
      "used_pdf": used_pdf
  }

In [27]:
# Call the agent with one question; it returns the dict built at the end of research_agent
result = research_agent("What are effective techniques to reduce hallucination in retrieval-augmented generation?")

print("\n" + "=" * 70)                                      # "=" * 70 draws a 70-character line
print(result["answer"])                                     # the cited answer + references
print("=" * 70)
print("Grounded:", result["grounded"], "| Rounds:", result["rounds"], "| Read PDFs:", result["used_pdf"])

Round 1: searching ['retrieval-augmented generation hallucination mitigation', 'hallucination reduction techniques RAG', 'knowledge grounding retrieval-augmented generation']
 6/8 papers judged relevant
  Verifier found problems -> rewriting once: The answer attributes a ~20% reduction in hallucination rates to the MMR approach described in source [2], but source [2] only reports a 15% improvement in retrieval accuracy and a 20% boost in context precision, mentioning a notable reduction in hallucination rates without quantifying it. Therefore the specific ~20% hallucination reduction claim is not directly supported by the source.

Effective mitigation of hallucination in Retrieval‑Augmented Generation (RAG) relies on a combination of retrieval‑level filtering, adaptive augmentation, and post‑generation verification.  Ensemble‑voting frameworks such as VOTE‑RAG improve robustness by (i) issuing multiple diverse queries in parallel and aggregating all retrieved passages (retrieval voting

In [30]:
result = research_agent(
    "What are the major approaches to improve retrieval quality in retrieval-augmented generation?"
)
print(result["answer"])

Round 1: searching ['retrieval-augmented generation retrieval quality', 'RAG retrieval improvement methods', 'enhancing retrieval in retrieval-augmented generation']
 6/8 papers judged relevant
Retrieval‑augmented generation (RAG) systems improve factual grounding by strengthening the retrieval component, and recent work converges on several complementary strategies.  

**Hybrid retrievers** combine sparse keyword matching (e.g., BM25) with dense semantic encoders (e.g., DPR) and learn a global weight balance, often via grid‑search optimisation, to raise recall across heterogeneous queries [2].  

**Recursive or hierarchical retrieval** parses long‑tail or multi‑granular documents and performs progressive, multi‑step retrievals that expand coverage of complex knowledge structures [2].  

**Iterative retrieval‑generation synergy** lets the language model’s intermediate output guide subsequent retrieval passes, creating a feedback loop where each generated fragment refines the query for 

In [31]:
result = research_agent(
    "How do dense retrieval and sparse retrieval methods compare for academic information retrieval?"
)
print(result["answer"])

Round 1: searching ['dense retrieval academic information retrieval', 'sparse retrieval academic IR', 'dense vs sparse retrieval methods']
 5/8 papers judged relevant
Dense retrieval and sparse retrieval represent two contrasting paradigms for academic information retrieval, each with distinct strengths and limitations. Sparse methods such as BM25 rely on high‑dimensional bag‑of‑words vectors and exact lexical matching, offering low computational cost and strong precision for queries that share surface terms with relevant papers [1]. Dense retrievers, built on transformer‑based encoders, map queries and documents into low‑dimensional embeddings, yielding substantial gains in semantic recall and overall effectiveness [1, 2]. However, dense models are computationally intensive, requiring GPUs and large memory footprints, and their fixed‑length encodings can struggle with long scholarly documents, limiting fine‑grained discrimination [2].

Empirical work shows that the two approaches retr